# LightGBM — r/Random_Acts_Of_Pizza request fulfillment

Binary classification, AUC-optimized. Hyperparameters from `CLAUDE.md` model notes.
Goal: beat the Logistic Regression baseline of **AUC 0.6452**.

Pipeline mirrors `xgboost.ipynb` so the two models can be compared on byte-identical splits and OOF predictions.

In [ ]:
# Setup: on Colab, clone the repo so utils.py is importable and install needed packages.
# No-op when running locally with the conda env.
import os, sys, subprocess

REPO_URL = "https://github.com/JaeHub/the-free-pizza-project"
REPO_BRANCH = "GB"
REPO_DIR = "/content/the-free-pizza-project"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)
    subprocess.run(["pip", "install", "-q", "kagglehub", "xgboost", "lightgbm"], check=True)

print("in_colab:", IN_COLAB, "| cwd:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

from utils import (
    TARGET_COLUMN,
    RANDOM_STATE,
    load_data,
    engineer_features,
    get_train_test_split,
    get_cv_folds,
    save_predictions,
    evaluate_cv,
    evaluate_test,
    plot_feature_importance,
    log_run,
)

MODEL_NAME = "lightgbm"
print("lightgbm version:", lgb.__version__)

## Load & engineer features

In [ ]:
df = load_data()
X = engineer_features(df)
y = df[TARGET_COLUMN].astype(int).values

print("X shape:", X.shape, "| target mean:", round(y.mean(), 4))
X.head()

## Train/test split (shared across all models)

In [ ]:
train_idx, test_idx = get_train_test_split(y)
X_train, X_test = X.iloc[train_idx].reset_index(drop=True), X.iloc[test_idx].reset_index(drop=True)
y_train, y_test = y[train_idx], y[test_idx]

print(f"train: {len(y_train)} (pos rate {y_train.mean():.3f})")
print(f"test:  {len(y_test)} (pos rate {y_test.mean():.3f})")

## 5-fold stratified CV with early stopping

Per CLAUDE.md, use `is_unbalance=True` (not both `is_unbalance` and `scale_pos_weight`).

In [ ]:
BASE_PARAMS = dict(
    objective="binary",
    metric="auc",
    num_leaves=31,
    learning_rate=0.05,
    n_estimators=1000,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    is_unbalance=True,
    random_state=RANDOM_STATE,
    verbosity=-1,
)

folds = get_cv_folds(y_train)
oof_probs = np.zeros(len(y_train), dtype=float)
best_iters = []

for fold_i, (tr_idx, val_idx) in enumerate(folds):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    model = lgb.LGBMClassifier(**BASE_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )

    oof_probs[val_idx] = model.predict_proba(X_val)[:, 1]
    best_iters.append(model.best_iteration_)
    fold_auc = roc_auc_score(y_val, oof_probs[val_idx])
    print(f"fold {fold_i}: best_iter={model.best_iteration_:4d}  val AUC={fold_auc:.4f}")

print("median best_iteration:", int(np.median(best_iters)))

In [ ]:
cv_results = evaluate_cv(y_train, oof_probs, MODEL_NAME)

## Refit on full train, evaluate once on held-out test

In [ ]:
n_folds = len(folds)
final_n_estimators = int(np.median(best_iters) * n_folds / (n_folds - 1))

FINAL_PARAMS = {**BASE_PARAMS, "n_estimators": final_n_estimators}
final_model = lgb.LGBMClassifier(**FINAL_PARAMS)
final_model.fit(X_train, y_train)

test_probs = final_model.predict_proba(X_test)[:, 1]
test_auc = evaluate_test(y_test, test_probs, MODEL_NAME)

## Save artifacts and inspect feature importance

In [ ]:
save_predictions(MODEL_NAME, oof_probs, test_probs)
importances = final_model.booster_.feature_importance(importance_type="gain")
plot_feature_importance(importances, X_train.columns, MODEL_NAME)
log_run(MODEL_NAME, FINAL_PARAMS, cv_results["mean"], cv_results["std"], test_auc)

## Summary

- **CV AUC**: see `cv_results["mean"] ± cv_results["std"]` above.
- **Test AUC**: see `test_auc` above.
- **Baselines**: Logistic Regression 0.6452 (target to beat); CNN 0.5846; XGBoost from `xgboost.ipynb`.
- **Cross-model fairness**: identical train/test indices and CV folds (cached in `splits/`); OOF arrays in `predictions/lightgbm_oof.npy` and `predictions/xgboost_oof.npy` are aligned positionally to `splits/train_idx.npy`.
- **Overfitting check**: if CV AUC ≪ refit train AUC, lower `num_leaves` or raise `min_child_samples` — but per CLAUDE.md, don't over-tune.